# Tutorial 05 — Post-training II: alignment (preference optimisation, RLHF and safety)

[Lecture notes](../lectures/05_alignment.md) · [Slides](../slides/05_alignment.pptx) · [Exercise sheet](../exercises/05_alignment.md)

**40 minutes, CPU, NumPy only.** You will minimise the DPO loss for a policy with two possible responses, see what disagreeing annotators do to it, compute group-relative advantages, and train a toy refusal policy on three preference sets to measure refusal and over-refusal. Everything here is synthetic. Nothing measures a real language model.

| Minutes | Step |
|---|---|
| 0–10 | DPO on a two-response policy |
| 10–15 | Annotators who disagree |
| 15–20 | Group-relative advantages |
| 20–30 | Refusal and over-refusal |
| 30–40 | Your turn: two functions with tests |

In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)

Python 3.13.1 NumPy 2.1.3 seed 42


## 1. DPO on a two-response policy

A prompt has two responses, A and B, and annotators prefer A. One number `theta` is the policy's log-odds of A over B, so $\Delta_\theta=\theta$. The reference policy is indifferent, so $\Delta_{ref}=0$.

Predict the loss before the first step, and whether P(A) reaches 1 in 50 steps.

In [2]:
def sigmoid(x): return 1/(1+np.exp(-x))
def dpo_loss(delta,reference,beta): return np.logaddexp(0,-beta*(delta-reference))
reference=0.0; theta=0.0; beta=2.0
assert np.isclose(dpo_loss(theta,reference,beta),np.log(2))
for step in range(51):
    z=beta*(theta-reference)
    grad=beta*(sigmoid(z)-1)
    theta-=.1*grad
    if step%10==0: print(step,"loss",dpo_loss(theta,reference,beta),"P(A)",sigmoid(theta))
assert sigmoid(theta)>.5

0 loss 0.5981388693815918 P(A) 0.52497918747894
10 loss 0.21510410081871448 P(A) 0.6711910596963976
20 loss 0.12266300827760743 P(A) 0.7346175540499525
30 loss 0.08449488967546502 P(A) 0.7710517790558927
40 loss 0.0640873839281572 P(A) 0.7953769206614584
50 loss 0.05148720776170229 P(A) 0.8131005364991026


The loss starts at $\log 2=0.693$ and is 0.051 after 50 steps, while P(A) has moved from 0.50 to 0.81. The steps shrink because the gradient is $\beta(\sigma(z)-1)$, which goes to zero as the pair is satisfied. Would P(A) rise faster or slower with `beta=0.5`?

## 2. Annotators who disagree

Ten annotators compare the same two responses. Seven prefer A and three prefer B. Note the two different probabilities: the model of the annotators' choice is $\sigma(\beta\theta)$, and the policy's probability of producing A is $\sigma(\theta)$.

In [3]:
labels=np.array([1]*7+[0]*3); theta_noisy=0.0
for _ in range(200):
    pred=sigmoid(beta*(theta_noisy-reference))
    theta_noisy-=.1*beta*(pred-labels).mean()
print("preference probability",sigmoid(beta*theta_noisy),"response probability",sigmoid(theta_noisy))
assert np.isclose(sigmoid(beta*theta_noisy),.7,atol=.01)

preference probability 0.6999999963392637 response probability 0.6043560741769496


Training stops where the preference probability equals the annotators' split, 0.70. The policy then produces A 60% of the time. With ten annotators who all agree there is no such stopping point, and `theta` grows for as long as you train.

## 3. Group-relative advantages

Sample four responses to one prompt and score each with a reward. The advantage of a response is its reward minus the group mean, divided by the group standard deviation. This is one ingredient of GRPO, and it is a long way from the full algorithm.

In [4]:
rewards=np.array([[0.,1.,1.,0.],[1.,1.,1.,1.]])
advantages=(rewards-rewards.mean(axis=1,keepdims=True))/(rewards.std(axis=1,keepdims=True)+1e-8)
print(advantages)
assert np.allclose(advantages.mean(axis=1),0)
assert np.allclose(advantages[1],0)
print("worked-example loss",dpo_loss(.4,.1,2))

[[-0.99999998  0.99999998  0.99999998 -0.99999998]
 [ 0.          0.          0.          0.        ]]
worked-example loss 0.4374879504858856


The second prompt had four correct responses and every advantage is zero, so it gives the policy no gradient. The same happens when all four are wrong. What does that imply for a task the model almost never solves?

## 4. Refusal and over-refusal

Each prompt is two numbers: how risky it sounds, and whether its intent is harmful. There are three kinds of prompt. Harmful prompts sound risky and are harmful. Plain safe prompts are neither. The third kind sounds risky and is safe, such as a question about how to kill a Python process; XSTest (Röttger et al., 2023) collects prompts of this kind.

The policy has two responses, refuse or comply, with log-odds `x @ w + b`. A preference pair says which response wins for a prompt. We train with the DPO loss against an indifferent reference on three preference sets and measure the share of held-out prompts refused.

Predict the three rows before running.

In [5]:
kinds = {"harmful": (1.5, 1.5), "plain safe": (-1.5, -1.5), "safe, sounds risky": (1.5, -1.5)}

def prompts(n, sounds_risky, harmful):
    return np.c_[rng.normal(sounds_risky, 1.0, n), rng.normal(harmful, 1.0, n)]

train = {name: prompts(200, *centre) for name, centre in kinds.items()}
test = {name: prompts(1000, *centre) for name, centre in kinds.items()}

def train_refusal_policy(names, beta=2.0, steps=300, lr=0.1):
    """DPO on pairs (refuse, comply). Refuse wins on harmful prompts; comply wins on the others."""
    X = np.r_[tuple(train[name] for name in names)]
    sign = np.r_[tuple(np.full(len(train[name]), 1.0 if name == "harmful" else -1.0) for name in names)]
    w, b = np.zeros(2), 0.0
    for _ in range(steps):
        delta = sign * (X @ w + b)                       # log-odds of the winning response; the reference has 0
        g = beta * (sigmoid(beta * delta) - 1) * sign
        w -= lr * (g[:, None] * X).mean(0); b -= lr * g.mean()
    return w, b

sets = {"refusals only": ["harmful"],
        "plus plain safe": ["harmful", "plain safe"],
        "plus safe that sounds risky": ["harmful", "plain safe", "safe, sounds risky"]}
refused = {}
print(f"{'preference set':30s}" + "".join(f"{name:>22s}" for name in kinds))
for label, names in sets.items():
    w, b = train_refusal_policy(names)
    refused[label] = [float((test[name] @ w + b > 0).mean()) for name in kinds]
    print(f"{label:30s}" + "".join(f"{rate:22.3f}" for rate in refused[label]))
assert refused["refusals only"][2] > 0.5 > refused["plus safe that sounds risky"][2]

preference set                               harmful            plain safe    safe, sounds risky
refusals only                                  0.999                 0.112                 0.881
plus plain safe                                0.976                 0.017                 0.559
plus safe that sounds risky                    0.919                 0.012                 0.079


Read the first column as the refusal rate on harmful prompts, which should be high, and the other two as over-refusal, which should be low. A preference set with refusals only gives 0.999 on harmful prompts and refuses 88% of the safe prompts that sound risky. Adding those prompts to the set, with "comply" as the winner, cuts over-refusal to 0.079, and the refusal rate on harmful prompts falls to 0.919. One number does not describe this policy. Report both rates.

## 5. Your turn

Replace each `raise` with your code, then run the test cell under it. Do not edit the tests.

**A. DPO loss from token log-probabilities.** A real model gives one log-probability per token. $\Delta$ is the summed log-probability of the winning response minus that of the losing response, over response tokens only. Each array has shape (pairs, tokens). A mask is 1 on response tokens and 0 on prompt and padding.

In [ ]:
# YOUR TURN
def sequence_dpo_loss(logp_w, logp_l, ref_w, ref_l, mask_w, mask_l, beta):
    """Mean DPO loss over the pairs: -log sigmoid(beta * (delta_theta - delta_ref)), with masked token sums."""
    raise NotImplementedError("your turn")

In [7]:
lw = np.array([[-0.5, -0.5, -50.0]]); mw = np.array([[1, 1, 0]])        # the last winner token is padding
ll = np.array([[-1.0, -1.0, -0.2]]);  ml = np.array([[1, 1, 1]])
rw = np.array([[-1.0, -1.0, -50.0]]); rl = np.array([[-1.0, -1.0, -0.2]])
# delta_theta = -1.0 - (-2.2) = 1.2, delta_ref = -2.0 - (-2.2) = 0.2, so the logit is 0.5 * 1.0
assert np.isclose(sequence_dpo_loss(lw, ll, rw, rl, mw, ml, 0.5), np.log(1 + np.exp(-0.5)))
assert np.isclose(sequence_dpo_loss(rw, rl, rw, rl, mw, ml, 3.0), np.log(2))      # policy equals reference
two = sequence_dpo_loss(np.r_[lw, rw], np.r_[ll, rl], np.r_[rw, rw], np.r_[rl, rl], np.r_[mw, mw], np.r_[ml, ml], 0.5)
assert np.isclose(two, (np.log(1 + np.exp(-0.5)) + np.log(2)) / 2)                # mean over pairs
print("loss for the worked pair:", round(float(sequence_dpo_loss(lw, ll, rw, rl, mw, ml, 0.5)), 4))
print("A passed")

loss for the worked pair: 0.4741
A passed


**B. Group advantages that survive a tie.** Standardise each row of rewards. A row whose rewards are all equal must return zeros, with no division by zero and no `nan`.

In [ ]:
# YOUR TURN
def group_advantages(rewards):
    """Row-wise (r - mean) / std. Rows with zero spread return all zeros."""
    raise NotImplementedError("your turn")

In [9]:
adv = group_advantages(np.array([[0., 1., 1., 0.], [1., 1., 1., 1.], [0., 0., 0., 0.], [2., 4., 6., 8.]]))
assert np.isfinite(adv).all()
assert np.allclose(adv[0], [-1, 1, 1, -1], atol=1e-6)
assert np.array_equal(adv[1], np.zeros(4)) and np.array_equal(adv[2], np.zeros(4))
assert np.allclose(adv[3].std(), 1, atol=1e-6) and np.isclose(adv[3].mean(), 0)
print(np.round(adv, 3))
print("B passed")

[[-1.     1.     1.    -1.   ]
 [ 0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.   ]
 [-1.342 -0.447  0.447  1.342]]
B passed


## Discussion

1. Can DPO lower the preference loss while making answers less factual?
2. Why does a group of identical rewards carry no signal for standardised group advantages?
3. What happens if the chosen and rejected labels are swapped in a preference file?
4. In section 4, what would a native speaker have to label before you could compute these two rates for Hausa prompts?

## From the toy to a real model

Collecting paired outputs from a real model needs `course/extensions/hf_text.py`, extra packages and a model download; read the [extension guide](../extensions/README.md) first. Label the pairs under a fixed written rubric, and keep the pairs you tune on apart from the pairs you report.

Submit this notebook with both tests passing and a 150–250-word note: which preference set from section 4 you would ship, the two rates that support the choice, and one way the toy flatters the method.